### Importing required libraries

In [1]:
from sklearn.model_selection import (
    train_test_split,
    cross_validate, 
    RandomizedSearchCV, 
    GridSearchCV, 
    StratifiedKFold
)

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier, 
)

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import shap
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

## Objective

This notebook builds and evaluates churn prediction models using the feature-engineered dataset prepared in `eda_feat_engg.ipynb`. The goals are:

- Define a clean feature set without target leakage
- Build preprocessing and modeling pipelines
- Compare baseline and advanced models
- Tune promising models
- Select a classification threshold using validation data
- Report final performance on an untouched test set
- Translate model outputs into a simple business scenario analysis


### Loading dataset

In [2]:
df = pd.read_csv("../data/processed.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df.head()


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason,tenure_duration,revenue_per_month,revenue_per_month_missing,revenue_deviation,revenue_deviation_missing
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,Yes,91,5433,Competitor,Competitor offered more data,0-12 (~1 Year),59.650000,0,1.504414,0
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,Yes,69,5302,Competitor,Competitor made better offer,0-12 (~1 Year),128.012500,0,1.587260,0
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,Yes,81,3179,Competitor,Competitor made better offer,12-24 (1-2 Years),106.160000,0,1.112205,0
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,Yes,88,5337,Dissatisfaction,Limited range of services,24-36 (2-3 Years),119.802800,0,1.216272,0
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,Yes,67,2793,Price,Extra data charges,36-48 (3-4 Years),83.847568,0,1.096047,0


The processed dataset includes the original variables plus engineered features from the EDA notebook. The preview below confirms that the loaded file contains 55 columns, including derived tenure and revenue features.

The processed dataset contains 7,043 customers and 55 columns. Compared with the raw dataset, it includes engineered features such as tenure bands and revenue-derived variables in addition to the original customer attributes.

In [3]:
leakage_columns = {
    'identifiers': ['Customer ID'],
    'target': ['Churn Label'],
    'post_outcome': [
        'Customer Status',
        'Churn Score',
        'Churn Category',
        'Churn Reason',
    ],
    'geographic': [
        'Country',
        'State',
        'City',
        'Zip Code',
        'Latitude',
        'Longitude',
    ],
    'additional': ['Satisfaction Score'],
}

all_leakage = sum(leakage_columns.values(), [])

print('Columns to exclude from modeling:')
print(all_leakage)

print('\nRemaining candidate features:')
candidate_features = [col for col in df.columns if col not in all_leakage]
print(candidate_features)

Columns to exclude from modeling:
['Customer ID', 'Churn Label', 'Customer Status', 'Churn Score', 'Churn Category', 'Churn Reason', 'Country', 'State', 'City', 'Zip Code', 'Latitude', 'Longitude', 'Satisfaction Score']

Remaining candidate features:
['Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Population', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'CLTV', 'tenure_duration', 'revenue_per_month', 'revenue_per_month_missing', 're

The excluded columns may still remain in the dataframe for bookkeeping, interpretation, or business evaluation. However, only the approved candidate features will be passed into the preprocessing and modeling pipeline.


## Train / validation / test split

The dataset is split into:

- Training set: 60%
- Validation set: 20%
- Test set: 20%

Stratification is used to preserve the churn rate in each split. The split is performed before preprocessing and model fitting. All preprocessing steps are later fit only on the training data within scikit-learn pipelines to prevent leakage.


In [4]:
feature_cols = [col for col in df.columns if col not in all_leakage]
X = df[feature_cols].copy()
y = df["Churn Label"].map({"No": 0, "Yes": 1})

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=7
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print("\nChurn rate in train:", y_train.mean().round(3))
print("Churn rate in val:", y_val.mean().round(3))
print("Churn rate in test:", y_test.mean().round(3))


Train shape: (4225, 42)
Validation shape: (1409, 42)
Test shape: (1409, 42)

Churn rate in train: 0.265
Churn rate in val: 0.265
Churn rate in test: 0.265


The following fields are retained separately for diagnostics and business simulation. They are not used as predictors in the churn model.


In [5]:
# Keep business-evaluation fields separately from the modeling feature matrix
business_cols = ["Customer ID", "CLTV", "Total Revenue"]

business_df = df.loc[X.index, business_cols].copy()

business_temp, business_test = train_test_split(
    business_df, test_size=0.2, stratify=y, random_state=7
)

business_train, business_val = train_test_split(
    business_temp, test_size=0.25, stratify=y_temp, random_state=7
)

train_ids = business_train["Customer ID"]
val_ids = business_val["Customer ID"]
test_ids = business_test["Customer ID"]

train_cltv = business_train["CLTV"]
val_cltv = business_val["CLTV"]
test_cltv = business_test["CLTV"]

train_revenue = business_train["Total Revenue"]
val_revenue = business_val["Total Revenue"]
test_revenue = business_test["Total Revenue"]


## Preprocessing pipeline

This section defines a preprocessing pipeline for numeric and categorical variables:

- Numeric features: median imputation followed by standard scaling
- Categorical features: constant-value imputation followed by one-hot encoding

All preprocessing is wrapped inside a scikit-learn `Pipeline` and `ColumnTransformer` so that transformations are learned only from the training data.


In [6]:
X_train_feat = X_train.copy()
X_val_feat = X_val.copy()
X_test_feat = X_test.copy()

num_features = X_train_feat.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train_feat.select_dtypes(exclude=[np.number]).columns.tolist()

print("Number of numeric features:", len(num_features))
print("Numeric features:", num_features)

print("\nNumber of categorical features:", len(cat_features))
print("Categorical features:", cat_features)


Number of numeric features: 18
Numeric features: ['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'CLTV', 'revenue_per_month', 'revenue_per_month_missing', 'revenue_deviation', 'revenue_deviation_missing']

Number of categorical features: 24
Categorical features: ['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Quarter', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'tenure_duration']


In [7]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_features),
        ("cat", categorical_pipeline, cat_features),
    ]
)


The preprocessing design makes the feature matrix robust to common data issues:

- Numeric variables use median imputation to reduce sensitivity to skew and outliers.
- Categorical variables use a constant `"Missing"` category so that missingness is preserved explicitly.
- One-hot encoding converts categorical values into a machine-readable representation.
- `handle_unknown="ignore"` ensures that new categories seen at validation or test time do not break the pipeline.


## Model comparison

This section compares several classification models:

- Logistic Regression
- Random Forest
- Decision Tree
- AdaBoost
- Gradient Boosting
- XGBoost
- CatBoost

Models are evaluated using cross-validation on the training set and then compared on the validation set. Reported metrics include ROC-AUC, PR-AUC, precision, recall, F1-score, balanced accuracy, and Brier score.


In [8]:
models = {
    'LogisticRegression': LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        random_state=7,
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight='balanced',
        random_state=7,
        n_jobs=-1,
    ),
    'DecisionTree': DecisionTreeClassifier(
        criterion='log_loss',
        max_depth=4,
        random_state=7,
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=300,
        random_state=7,
    ),
    'GradientBoost': GradientBoostingClassifier(
        random_state=7,
    ),
    'XGB': XGBClassifier(
        random_state=7,
    ),
    'CatBoost': CatBoostClassifier(
        verbose=0,
        random_seed=7,
    ),
}

def evaluate_model(model, X_train, y_train, X_val, y_val):
    pipe = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', model),
        ]
    )

    pipe.fit(X_train, y_train)

    y_val_proba = pipe.predict_proba(X_val)[:, 1]
    y_val_pred = pipe.predict(X_val)

    metrics = {
        'roc_auc': roc_auc_score(y_val, y_val_proba),
        'pr_auc': average_precision_score(y_val, y_val_proba),
        'precision': precision_score(y_val, y_val_pred, zero_division=0),
        'recall': recall_score(y_val, y_val_pred, zero_division=0),
        'f1': f1_score(y_val, y_val_pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_val, y_val_pred),
        'brier_score': brier_score_loss(y_val, y_val_proba),
    }

    return pipe, metrics

cv_results = []

for name, model in models.items():
    pipe = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ]
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

    scores = cross_validate(
        pipe, 
        X_train_feat,
        y_train,
        cv=cv,
        scoring=['roc_auc', 'average_precision'],
        n_jobs=-1,
    )

    cv_results.append({
        'model': name,
        'cv_roc_auc_mean': scores['test_roc_auc'].mean(),
        'cv_roc_auc_std': scores['test_roc_auc'].std(),
        'cv_pr_auc_mean': scores['test_average_precision'].mean(),
        'cv_pr_auc_std': scores['test_average_precision'].std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values('cv_pr_auc_mean', ascending=False)

display(cv_results_df.round(4))

,model,cv_roc_auc_mean,cv_roc_auc_std,cv_pr_auc_mean,cv_pr_auc_std
6,CatBoost,0.9119,0.0095,0.7916,0.0315
4,GradientBoost,0.9104,0.0098,0.7880,0.0298
0,LogisticRegression,0.9053,0.0069,0.7747,0.0225
3,AdaBoost,0.9026,0.0112,0.7675,0.0336
5,XGB,0.9018,0.0112,0.7653,0.0353
1,RandomForest,0.8993,0.0100,0.7601,0.0359
2,DecisionTree,0.8634,0.0165,0.6402,0.0302


Cross-validation summarizes average training-set performance stability across folds, while the validation set provides an independent holdout check for model selection. Both are useful: cross-validation reduces dependence on a single split, and validation performance helps confirm how well each fitted pipeline generalizes.

In [9]:
val_results = []

for name, model in models.items():
    pipe, metrics = evaluate_model(
        model,
        X_train_feat,
        y_train,
        X_val_feat,
        y_val,
    )

    val_results.append({'model': name, **metrics})

val_results_df = pd.DataFrame(val_results).sort_values('pr_auc', ascending=False)

display(val_results_df.round(4))

,model,roc_auc,pr_auc,precision,recall,f1,balanced_accuracy,brier_score
6,CatBoost,0.8970,0.7675,0.7139,0.6738,0.6933,0.7881,0.1123
4,GradientBoost,0.8950,0.7586,0.7066,0.6631,0.6841,0.7818,0.1131
0,LogisticRegression,0.8927,0.7463,0.5540,0.8636,0.6750,0.8062,0.1446
1,RandomForest,0.8829,0.7377,0.6290,0.7433,0.6814,0.7924,0.1270
3,AdaBoost,0.8867,0.7357,0.6803,0.6658,0.6730,0.7764,0.1894
5,XGB,0.8807,0.7184,0.6851,0.6283,0.6555,0.7620,0.1323
2,DecisionTree,0.8398,0.5873,0.6043,0.6043,0.6043,0.7306,0.1398


Among the baseline models, CatBoost and Gradient Boosting provide the strongest overall discrimination, while Logistic Regression remains competitive and more interpretable. Logistic Regression achieves high recall but lower precision, whereas CatBoost offers a more balanced trade-off and stronger probability quality based on Brier score.


## Hyperparameter Tuning

This section performs hyperparameter tuning for selected models using RandomizedSearchCV with stratified cross-validation on the training set. The search uses PR-AUC (average_precision) as the primary scoring metric.

**Logistic Regression search space**

In [10]:
log_reg = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',
    random_state=7,
)

log_reg_param_dist = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 20, 50, 100, 1000],
}

**Gradient Boost search space**

In [11]:
gb = GradientBoostingClassifier(
    loss='log_loss',
    random_state=7
)

gb_param_dist = {
    'classifier__n_estimators': [200, 400, 600],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.7, 0.8, 1.0],
}


**Cat Boost search space**

In [12]:
cb = CatBoostClassifier(
    eval_metric='Logloss',
    random_seed=7,
    verbose=0,
)

cb_param_dist = {
    'classifier__n_estimators': [200, 400, 600],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.7, 0.8, 1.0],
    #'classifier__colsample_bytree': [0.7, 0.8, 1.0],
    #'classifier__min_child_weight': [1, 3, 5],
}

In [13]:
def make_clf_pipeline(classifier):
    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', classifier),
        ]
    )

log_reg_pipe = make_clf_pipeline(log_reg)
gb_pipe = make_clf_pipeline(gb)
cb_pipe = make_clf_pipeline(cb)

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
}

**Running GridSearch/RandomizedSearch CV for each model**

*Logistic Regression tuning*

In [15]:
log_reg_search = GridSearchCV(
    estimator=log_reg_pipe,
    param_grid=log_reg_param_dist,
    scoring=scoring,
    refit='pr_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

log_reg_search.fit(X_train_feat, y_train)

print('Best Logistic Regression params:')
print(log_reg_search.best_params_)
print('Best CV PR-AUC:', log_reg_search.best_score_)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Logistic Regression params:
{'classifier__C': 10}
Best CV PR-AUC: 0.775957503717105


*Gradient Boost  tuning*

In [16]:
gb_search = RandomizedSearchCV(
    estimator=gb_pipe,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring=scoring,
    refit='pr_auc',
    cv=cv,
    random_state=7,
    verbose=1,
    n_jobs=-1
)

gb_search.fit(X_train_feat, y_train)

print('Best Gradient Boost params:')
print(gb_search.best_params_)
print('Best CV PR-AUC:', gb_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Gradient Boost params:
{'classifier__subsample': 0.7, 'classifier__n_estimators': 200, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.05}
Best CV PR-AUC: 0.7882886443363497


*Cat Boost tuning*

In [17]:
cb_search = RandomizedSearchCV(
    estimator=cb_pipe,
    param_distributions=cb_param_dist,
    n_iter=50,
    scoring=scoring,
    refit='pr_auc',
    cv=cv,
    random_state=7,
    verbose=1,
    n_jobs=-1,
)

cb_search.fit(X_train_feat, y_train)

print('Best Cat Boost params:')
print(cb_search.best_params_)
print('Best CV PR-AUC:', cb_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Cat Boost params:
{'classifier__subsample': 0.7, 'classifier__n_estimators': 200, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.05}
Best CV PR-AUC: 0.799385667441844


The tuning ranges focus on a small number of influential hyperparameters to balance model quality and computational cost. The objective is not exhaustive search, but a controlled refinement of the most promising baseline models.

### Comparing tuned models on validation set

In [18]:
tuned_models = {
    'LogisticRegression': log_reg_search.best_estimator_,
    'GradientBoost': gb_search.best_estimator_,
    'CatBoost': cb_search.best_estimator_,
}

In [19]:
def evaluate_pipeline(pipe, X_val, y_val, threshold=0.5):
    y_val_proba = pipe.predict_proba(X_val)[:, 1]
    y_val_pred = (y_val_proba >= threshold).astype(int)

    metrics = {
        'roc_auc': roc_auc_score(y_val, y_val_proba),
        'pr_auc': average_precision_score(y_val, y_val_proba),
        'precision': precision_score(y_val, y_val_pred, zero_division=0),
        'recall': recall_score(y_val, y_val_pred, zero_division=0),
        'f1': f1_score(y_val, y_val_pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_val, y_val_pred),
        'brier_score': brier_score_loss(y_val, y_val_proba),
    }

    return metrics, y_val_proba

In [20]:
val_tuned_results = []

for name, pipe in tuned_models.items():
    metrics, _ = evaluate_pipeline(pipe, X_val_feat, y_val)
    val_tuned_results.append({'model': name, **metrics})

val_tuned_df = pd.DataFrame(val_tuned_results).sort_values('pr_auc', ascending=False)

display(val_tuned_df.round(4))

,model,roc_auc,pr_auc,precision,recall,f1,balanced_accuracy,brier_score
2,CatBoost,0.9006,0.7738,0.7275,0.6711,0.6982,0.7902,0.1096
1,GradientBoost,0.8942,0.7595,0.7020,0.6551,0.6777,0.7773,0.1134
0,LogisticRegression,0.8933,0.7485,0.5552,0.8610,0.6751,0.8058,0.1445


In [21]:
best_model_name = val_tuned_df.iloc[0]['model']
best_pipe = tuned_models[best_model_name]

print('Selected model:', best_model_name)

Selected model: CatBoost


## Threshold selection

For imbalanced classification, the default probability threshold of 0.50 is not automatically optimal. Different thresholds produce different trade-offs between precision, recall, and the number of customers flagged for intervention.

The appropriate threshold depends on business priorities such as intervention capacity, acceptable false-positive cost, and the value of recovering true churners.


In [22]:
# Use the tuned best model from validation 
y_val_proba = best_pipe.predict_proba(X_val_feat)[:, 1]

threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.05):
    y_val_pred = (y_val_proba >= threshold).astype(int)

    threshold_results.append({
        'threshold': threshold,
        'precision': precision_score(y_val, y_val_pred, zero_division=0),
        'recall': recall_score(y_val, y_val_pred, zero_division=0),
        'f1': f1_score(y_val, y_val_pred, zero_division=0),
        'customers_flagged': y_val_pred.sum(),
        'flagged_rate': y_val_pred.mean(),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(3))

,threshold,precision,recall,f1,customers_flagged,flagged_rate
0,0.05,0.412,0.987,0.582,895,0.635
1,0.10,0.462,0.957,0.623,775,0.550
2,0.15,0.494,0.928,0.644,703,0.499
3,0.20,0.525,0.888,0.660,632,0.449
4,0.25,0.566,0.856,0.682,565,0.401
5,0.30,0.605,0.824,0.698,509,0.361
6,0.35,0.640,0.802,0.712,469,0.333
7,0.40,0.662,0.759,0.707,429,0.304
8,0.45,0.688,0.725,0.706,394,0.280
9,0.50,0.728,0.671,0.698,345,0.245


The threshold table shows the operational trade-off clearly: lower thresholds increase recall but flag more customers, while higher thresholds improve precision but miss more churners. The final threshold should therefore be justified explicitly in terms of business goals rather than chosen mechanically.


In [23]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=threshold_df['threshold'],
        y=threshold_df['precision'],
        mode='lines+markers',
        name='Precision',
        line=dict(width=3, color='#1f77b4'),
        marker=dict(size=8),
    )
)

fig.add_trace(
    go.Scatter(
        x=threshold_df['threshold'],
        y=threshold_df['recall'],
        mode='lines+markers',
        name='Recall',
        line=dict(width=3, color='#ff7f0e'),
        marker=dict(size=8),
    )
)

fig.add_trace(
    go.Scatter(
        x=threshold_df['threshold'],
        y=threshold_df['f1'],
        mode='lines+markers',
        name='F1',
        line=dict(width=3, color='#2ca02c'),
        marker=dict(size=8),
    )
)

fig.update_layout(
    title='Precision, Recall, and F1 vs Threshold',
    xaxis_title='Threshold',
    yaxis_title='Metric',
    template='plotly_white',
    title_x=0.5,
    hovermode='x unified',
    legend_title_text='Metric',
    width=700,
    height=500,
)

fig.update_yaxes(range=[0, 1])

fig.show()

This chart shows how model performance changes as the classification threshold is adjusted. Lower thresholds classify more customers as likely churners, which usually increases recall but reduces precision. Higher thresholds do the opposite, making the model more selective and typically improving precision while reducing recall.

The F1 score summarizes the balance between precision and recall. The threshold where F1 is highest is often a good default operating point when both false positives and false negatives matter, although the final choice should also reflect business constraints such as campaign budget and contact capacity.

This plot is useful because churn prediction is not only about ranking customers, but also about deciding where to set the cutoff for action. A retention team with limited budget may prefer a higher threshold to target fewer but more likely churners, while a team focused on catching as many potential churners as possible may accept a lower threshold and lower precision.


In [24]:
idx_best_f1 = threshold_df['f1'].idxmax()
selected_threshold = threshold_df.loc[idx_best_f1, 'threshold']

print('Selected threshold (max F1):', selected_threshold)
print(threshold_df.loc[idx_best_f1])

Selected threshold (max F1): 0.35000000000000003
threshold              0.350000
precision              0.639659
recall                 0.802139
f1                     0.711744
customers_flagged    469.000000
flagged_rate           0.332860
Name: 6, dtype: float64


### Selected threshold

The final threshold selected for downstream evaluation is 0.35. It was chosen because it provides the preferred balance between recall, precision, and intervention volume on the validation set.

This means that any customer with predicted churn probability greater than or equal to this threshold is flagged for intervention.


## Final test-set evaluation

The model and threshold are selected using training and validation data only. The test set is used once, for final evaluation.

In [25]:
# Get the underlying classifier type and best params
if best_model_name == 'LogisticRegression':
    best_clf = LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        random_state=7,
        **{k.replace('classifier__', ''): v
           for k, v in log_reg_search.best_params_.items()},
    )
elif best_model_name == 'GradientBoost':
    best_clf = RandomForestClassifier(
        class_weight='balanced',
        random_state=7,
        n_jobs=-1,
        **{k.replace('classifier__', ''): v
           for k, v in gb_search.best_params_.items()},
    )
elif best_model_name == 'CatBoost':
    best_clf = XGBClassifier(
        eval_metric='logloss',
        random_state=7,
        n_jobs=-1,
        **{k.replace('classifier__', ''): v
           for k, v in cb_search.best_params_.items()},
    )

final_pipe = make_clf_pipeline(best_clf)

X_train_val = pd.concat([X_train_feat, X_val_feat], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

final_pipe.fit(X_train_val, y_train_val)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](42,)","['Gender','Age','Under 30',...,'revenue_per_month_missing', 'revenue_deviation','revenue_deviation_missing']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,42
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By sp

In [26]:
y_test_proba = final_pipe.predict_proba(X_test_feat)[:, 1]
y_test_pred = (y_test_proba >= selected_threshold).astype(int)

test_metrics = {
    'roc_auc': roc_auc_score(y_test, y_test_proba),
    'pr_auc': average_precision_score(y_test, y_test_proba),
    'precision': precision_score(y_test, y_test_pred, zero_division=0),
    'recall': recall_score(y_test, y_test_pred, zero_division=0),
    'f1': f1_score(y_test, y_test_pred, zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'brier_score': brier_score_loss(y_test, y_test_proba),
    'customers_flagged': y_test_pred.sum(),
    'flagged_rate': y_test_pred.mean(),
}

display(pd.Series(test_metrics).round(4))

roc_auc                0.9020
pr_auc                 0.7930
precision              0.6667
recall                 0.7807
f1                     0.7192
balanced_accuracy      0.8198
brier_score            0.1064
customers_flagged    438.0000
flagged_rate           0.3109
dtype: float64

In [27]:
result = pd.DataFrame({
    'actual': y_test,
    'prediction': y_test_pred,
    'pred_proba': y_test_proba,
})

cm = confusion_matrix(result['actual'], result['prediction'])
cm_pct = cm / cm.sum()

cm_df = pd.DataFrame(
    cm,
    index=['Actual: Not Churn', 'Actual: Churn'],
    columns=['Predicted: Not Churn', 'Predicted: Churn'],
)

text_matrix = np.array([
    [f'{cm[i, j]}<br>({cm_pct[i, j]:.1%})' for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
])

hover_matrix = np.array([
    [f'{cm[i, j]:,} ({cm_pct[i, j]:.1%})' for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
])


fig = px.imshow(
    cm_df,
    aspect='equal',
    color_continuous_scale=[
        [0.0, '#f7fbff'],
        [0.2, '#deebf7'],
        [0.4, '#c6dbef'],
        [0.6, '#9ecae1'],
        [0.8, '#6baed6'],
        [1.0, '#2171b5'],
    ],
)

fig.update_traces(
    text=text_matrix,
    customdata=hover_matrix,
    texttemplate='%{text}',
    textfont=dict(size=18, color='black'),
    hovertemplate=(
        '%{y}<br>%{x}<br>'
        '%{customdata}<extra></extra>'
    ),
)

fig.update_layout(
    title=dict(
        text='Confusion Matrix (Test Set)',
        x=0.5,
        xanchor='center',
        font=dict(size=22),
    ),
    width=700,
    height=560,
    template='plotly_white',
    margin=dict(l=40, r=40, t=90, b=40),
    coloraxis_colorbar=dict(
        title='Count',
        thickness=16,
        len=0.75,
    ),
)

fig.update_xaxes(
    title_text='Predicted class',
    tickfont=dict(size=13),
    title_font=dict(size=15),
)

fig.update_yaxes(
    title_text='Actual class',
    tickfont=dict(size=13),
    title_font=dict(size=15),
)

fig.show()

## Calibration

This section evaluates how well predicted probabilities match observed churn rates.

In [28]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(
    y_test,
    y_test_proba,
    n_bins=10,
    strategy='quantile',
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=prob_pred,
        y=prob_true,
        mode='lines+markers',
        name='Model',
        line=dict(color="#326AB9", width=3),
        marker=dict(size=8),
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode='lines',
        name='Perfectly calibrated',
        line=dict(color='black', dash='dash', width=2),
    )
)

fig.update_layout(
    title='Calibration Curve (Test Set)',
    xaxis_title='Mean predicted probabilities',
    yaxis_title='Fraction of positives',
    template='plotly_white',
    title_x=0.5,
    hovermode='closest',
    width=650,
    height=550,
)

fig.update_xaxes(range=[0, 1], tickformat='.0%')
fig.update_yaxes(range=[0, 1], tickformat='.0%')

fig.show()

**Interpretation**

This calibration curve compares the model’s predicted churn probabilities with the actual observed churn rates in the test set. Each point represents a bin of customers grouped by similar predicted probabilities.

If the model is perfectly calibrated, the curve will follow the diagonal line. Points above the diagonal indicate that the observed churn rate is higher than the predicted probability, meaning the model is underconfident in that range. Points below the diagonal indicate that the model is overconfident, assigning probabilities that are higher than the true churn rate.

Good calibration is important because this project uses predicted probabilities for business decisions such as threshold selection, campaign targeting, and expected value calculations. Even a model with strong ranking performance can be misleading if its probability estimates are poorly calibrated.

Overall, this plot should be interpreted as a probability-quality diagnostic. It complements discrimination metrics such as ROC-AUC and PR-AUC by showing whether the predicted risk scores can be trusted as actual churn likelihoods.


## Model interpretability with SHAP

SHAP values describe how each feature contributes to the model’s predicted churn risk. Positive SHAP values increase the predicted probability of churn, while negative values reduce it.

These explanations describe the behavior of the fitted model; they should not be interpreted as causal effects in the real world.


This section uses SHAP to summarize both global feature importance and the direction of feature contributions. The goal is to understand what drives the model’s predictions, not to claim that these variables cause churn.

In [29]:
# Transform test data through the preprocessor
X_test_transformed = final_pipe.named_steps['preprocessor'].transform(X_test_feat)
feature_names = final_pipe.named_steps['preprocessor'].get_feature_names_out()

# Wrapper that returns predicted probabilities for the positive class
def model_predict_proba(X):
    return final_pipe.named_steps['classifier'].predict_proba(X)[:, 1]

# Use a subset for speed
X_sample = X_test_transformed[:500]

# Background data for explainer
background = shap.maskers.Independent(X_sample, max_samples=min(200, len(X_sample)))

explainer = shap.Explainer(
    model_predict_proba,
    masker=background,
    feature_names=feature_names,
)

shap_values = explainer.shap_values(X_sample)  # shape: (n_samples, n_features)

# Mean absolute SHAP for ordering
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
# Mean SHAP (with sign) for coloring
mean_shap = np.mean(shap_values, axis=0)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap,
    'mean_shap': mean_shap,
}).sort_values('mean_abs_shap', ascending=True)

top_n = 20
importance_df = importance_df.tail(top_n).copy()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=importance_df['mean_abs_shap'],
    y=importance_df['feature'],
    orientation='h',
    marker=dict(
        color=importance_df['mean_shap'],
        colorscale='RdBu',
        cmin=-max(np.abs(importance_df['mean_shap'])),
        cmax= max(np.abs(importance_df['mean_shap'])),
        showscale=True,
        colorbar=dict(title='Mean SHAP'),
    ),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Mean |SHAP|: %{x:.4f}<br>'
        'Mean SHAP: %{marker.color:.4f}<extra></extra>'
    ),
))

fig.update_layout(template='plotly_white',
    title='Global feature importance (mean |SHAP|) colored by mean SHAP',
    title_x=0.5,
    xaxis_title='Mean |SHAP value|',
    yaxis_title='Feature',
    height=min(600, 40 + top_n * 25),
    margin=dict(l=220, r=40, t=60, b=20),
)

fig.show()

PermutationExplainer explainer: 501it [00:15, 11.88it/s]                         


SHAP values describe feature contributions to the model’s predictions. They
should not be interpreted as causal effects.

## Business simulation: revenue impact of churn modeling

This section translates the classification results into a simple intervention scenario. The idea is to estimate how much customer value might be preserved if the model is used to target retention actions.

Key assumptions:

- All customers predicted as churners are contacted.
- Each contacted customer incurs a fixed intervention cost.
- Only true positives can generate retained value in this simplified setup.
- Only a fraction of contacted true positives are assumed to stay, captured by a retention-rate parameter.
- Retained value is approximated using `CLTV` or `Total Revenue` as a proxy.

This is a scenario analysis, not a causal estimate of realized profit. It ignores many operational details such as heterogeneous treatment effects, offer costs, customer behavior changes, and long-run dynamics.

In [30]:
# Prediction on test set
y_test_proba = final_pipe.predict_proba(X_test_feat)[:, 1]
y_test_pred = (y_test_proba >= selected_threshold).astype(int)

# Attach predictions to test dataframe
df_test = df.loc[X_test_feat.index].copy()
df_test['y_true'] = y_test.values
df_test['y_pred'] = y_test_pred
df_test['y_proba'] = y_test_proba

# True positives: actual churners that we predicted as churners
df_test['TP'] = ((df_test['y_true'] == 1) & (df_test['y_pred'] == 1)).astype(int)

# For clarity, also defining other groups
df_test['FP'] = ((df_test['y_true'] == 0) & (df_test['y_pred'] == 1)).astype(int)
df_test['TN'] = ((df_test['y_true'] == 0) & (df_test['y_pred'] == 0)).astype(int)
df_test['FN'] = ((df_test['y_true'] == 1) & (df_test['y_pred'] == 0)).astype(int)

In [31]:
# Scenario assumptions (can be varied later in sensitivity analysis)
cost_per_contact = 10.0
retention_rate = 1.0  # optimistic upper-bound scenario

# Choosing CLTV as the monetary metric, later will be done for total revenue as well
money_column = ['CLTV', 'Total Revenue']

# Ensure the column is present in the dataset
assert money_column[0] in df_test.columns, f'{money_column} not found in test dataframe'
assert money_column[1] in df_test.columns, f'{money_column} not found in test dataframe' 

In [32]:
# Baseline: no modeling, no intervention
baseline_contacted = 0
baseline_cost = 0
baseline_retained_churners = 0
baseline_saved_money = 0.0

# Total at-risk money in the test set (all churners)
total_churners = (df_test['y_true'] == 1).sum()
total_at_risk_CLTV = df_test.loc[df_test['y_true'] == 1, money_column[0]].sum()
total_at_risk_revenue = df_test.loc[df_test['y_true'] == 1, money_column[1]].sum()
print(total_churners)
print(total_at_risk_CLTV)
print(total_at_risk_revenue)

374
1559791
765664.96


In [33]:
# Customers we contact: all predicted churners
contacted_mask = df_test['y_pred'] == 1
n_contacted = contacted_mask.sum()

# Contact cost
total_contact_cost = n_contacted * cost_per_contact

# True positive among contacted
tp_mask = (df_test['TP'] == 1)
n_true_positives = tp_mask.sum()

# Among true positives, only a fraction are retained due to intervention
n_retained_churners = int(np.round(n_true_positives * retention_rate))

# Money associated with true positives (as-risk money we have a chance to save)
tp_CLTV = df_test.loc[tp_mask, money_column[0]].sum()
tp_revenue = df_test.loc[tp_mask, money_column[1]].sum()

# Average money per true positive churner
avg_CLTV_per_tp = tp_CLTV / n_true_positives if n_true_positives > 0 else np.nan
avg_revenue_per_tp = tp_revenue / n_true_positives if n_true_positives > 0 else np.nan

# Saved money: only from retained true positives
saved_CLTV = n_retained_churners * avg_CLTV_per_tp
saved_revenue = n_retained_churners * avg_revenue_per_tp

# Net benefit: saved money minus the contacted cost
net_benefit_CLTV = saved_CLTV - total_contact_cost
net_benefit_revenue = saved_revenue - total_contact_cost

# Cost per retained customer
cost_per_retained = total_contact_cost / n_retained_churners if n_retained_churners > 0 else np.nan

# ROI: Each dollar spent yields approximately
roi_CLTV = (saved_CLTV - total_contact_cost) / total_contact_cost if total_contact_cost > 0 else np.nan
roi_revenue = (saved_revenue - total_contact_cost) / total_contact_cost if total_contact_cost > 0 else np.nan

In [34]:
summary = {
    'Metric': [
        'Test set size',
        'Actual churners (y_true=1)',
        'Predicted churners (y_pred=1)',
        'True positives (TP)',
        'Initial CLTV at risk ($)',
        'Initial total revenue at risk ($)',
        'Final CLTV churned ($)',
        'Final Revenue churned ($)',
        'Contact cost per customer ($)',
        'Retention rate (of contacted TP)',
        f'Total contact cost ($)',
        f'Estimated retained churners',
        f'Saved {money_column[0]} ($)',
        f'Saved {money_column[1]} ($)',
        f'Net benefit ({money_column[0]} scenario, $)',
        f'Net benefit ({money_column[1]} scenario, $)',
        f'ROI ({money_column[0]} scenario)',
        f'ROI ({money_column[1]} scenario)',
    ],
    'Value': [
        len(df_test),
        int(total_churners),
        int(n_contacted),
        int(n_true_positives),
        total_at_risk_CLTV,
        total_at_risk_revenue,
        total_at_risk_CLTV - saved_CLTV,
        total_at_risk_revenue - saved_revenue,
        cost_per_contact,
        f'{retention_rate:.0%}',
        total_contact_cost,
        n_retained_churners,
        saved_CLTV,
        saved_revenue,
        net_benefit_CLTV,
        net_benefit_revenue,
        f'{roi_CLTV:.2f}' if not np.isnan(roi_CLTV) else 'N/A',
        f'{roi_revenue:.2f}' if not np.isnan(roi_revenue) else 'N/A',
    ],
}

summary_df = pd.DataFrame(summary)
display(summary_df)


,Metric,Value
0,Test set size,1409
1,Actual churners (y_true=1),374
2,Predicted churners (y_pred=1),438
3,True positives (TP),292
4,Initial CLTV at risk ($),1559791
5,Initial total revenue at risk ($),765664.96
6,Final CLTV churned ($),396371.0
7,Final Revenue churned ($),275625.13
8,Contact cost per customer ($),10.0
9,Retention rate (of contacted TP),100%


These results should be interpreted as an upper-bound or scenario-based estimate of value, not as guaranteed financial impact. They are most useful for comparing intervention assumptions, thresholds, and contact-cost levels rather than for claiming exact expected profit.


In [35]:
cltv_chart_df = pd.DataFrame({
    'Stage': [
        'Initial CLTV at risk ($)',
        'Final CLTV churned ($)',
    ],
    'Value': [
        total_at_risk_CLTV,
        total_at_risk_CLTV - saved_CLTV,
    ]
})

fig_cltv = px.bar(
    cltv_chart_df,
    x='Stage',
    y='Value',
    text='Value',
    color='Stage',
    color_discrete_sequence=['#D83C28',"#E8968B"]
)

fig_cltv.update_traces(
    texttemplate='%{text:,.2f}',
    textposition='outside'
)

fig_cltv.update_layout(template='plotly_white',
    title=dict(text='Churn CLTV: Before vs After modeling', x=0.5),
    showlegend=False,
    width=700,
    height=500,
    #plot_bgcolor='white',
    yaxis_title='CLTV',
    xaxis_title=''
)

fig_cltv.show()

In [36]:
revenue_chart_df = pd.DataFrame({
    'Stage': [
        'Initial Total Revenue at risk ($)',
        'Final Total Revenue churned ($)',
    ],
    'Value': [
        total_at_risk_revenue,
        total_at_risk_revenue - saved_revenue,
    ]
})

fig_revenue = px.bar(
    revenue_chart_df,
    x='Stage',
    y='Value',
    text='Value',
    color='Stage',
    color_discrete_sequence=['#D83C28',"#E8968B"]
)

fig_revenue.update_traces(
    texttemplate='%{text:,.2f}',
    textposition='outside'
)

fig_revenue.update_layout(template='plotly_white',
    title=dict(text='Churn Revenue: Before vs After modeling', x=0.5),
    showlegend=False,
    width=700,
    height=500,
    #plot_bgcolor='white',
    yaxis_title='Revenue',
    xaxis_title=''
)

fig_revenue.show()

In [37]:
saved_chart_df = pd.DataFrame({
    'Metric': ['Saved CLTV', 'Saved Total Revenue'],
    'Value': [saved_CLTV, saved_revenue]
})

fig_saved = px.bar(
    saved_chart_df,
    x='Metric',
    y='Value',
    text='Value',
    color='Metric',
    color_discrete_sequence=["#1B6619","#6BA369"]
)

fig_saved.update_traces(
    texttemplate='%{text:,.2f}',
    textposition='outside'
)

fig_saved.update_layout(template='plotly_white',
    title=dict(text='Business Value Saved by Model', x=0.5),
    showlegend=False,
    width=700,
    height=500,
    #plot_bgcolor='white',
    yaxis_title='Value',
    xaxis_title=''
)

fig_saved.show()

### Interpretation: before vs after modeling

- **Before modeling:**
  - No customers are proactively contacted.
  - All 374 (`total_churners`) churners are assumed to leave.
  - Total at-risk CLTV: $1,559,791.00 (`total_at_risk_CLTV`)
  - Total at-risk revenue: $765,664.96 (`total_at_risk_revenue`)
  - Saved CLTV: $0.00
  - Saved Total Revenue: $0.00

- **After modeling:**
  - We contact all customers predicted as churners (`n_contacted`).
  - Among them, 292 (`n_true_positives`) are true churners.
  - Assuming a retention rate of 100% {retention_rate:.0%}, about `n_retained_churners` churners are retained.
  - Estimated saved CLTV: $1,163,420.00
  - Estimated saved Total Revenue: $490,039.83
  - Total contact cost: $4,380.
  - Net benefit (CLTV): $1,159,040.00
  - Net benefit (Total Revenue): $485,659.83
  - ROI (CLTV): 264.62
  - ROI (Total Revenue): 110.88


In [38]:
# Simple comparison table
before_after = {
    'Scenario': ['Before modeling', 'After modeling'],
    'Customers contacted': [0, n_contacted],
    'Retained churners': [0, n_retained_churners],
    f'Saved {money_column[0]} ($)': [0, saved_CLTV],
    f'Saved {money_column[1]} ($)': [0, saved_revenue],
    'Contacted cost ($)': [0, total_contact_cost],
    'Net benefit (CLTV) ($)': [0, net_benefit_CLTV],
    'Net benefit (Total Revenue) ($)': [0, net_benefit_revenue],
}

before_after_df = pd.DataFrame(before_after)
display(before_after_df)

,Scenario,Customers contacted,Retained churners,Saved CLTV ($),Saved Total Revenue ($),Contacted cost ($),Net benefit (CLTV) ($),Net benefit (Total Revenue) ($)
0,Before modeling,0,0,0.0,0.00,0.0,0.0,0.00
1,After modeling,438,292,1163420.0,490039.83,4380.0,1159040.0,485659.83


## Sensitivity analysis

The business case depends strongly on assumptions about intervention cost and retention effectiveness. To test this, I evaluate a grid of scenarios across multiple retention rates and fixed contact costs.

The resulting `sensitivity_df` records how net benefit changes under each combination. This helps identify break-even regions and shows whether the intervention remains attractive under more conservative assumptions.


In [39]:
retention_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
costs = [5, 10, 20]

rows = []

for r in retention_rates:
    for c in costs:
        retained = int(np.round(n_true_positives * r))
        saved_cltv = retained * avg_CLTV_per_tp
        saved_rev = retained * avg_revenue_per_tp
        cost = n_contacted * c
        net_cltv = saved_cltv - cost
        net_rev = saved_rev - cost
        roi_val_cltv = (saved_cltv - cost) / cost if cost > 0 else np.nan
        roi_val_rev = (saved_rev - cost) / cost if cost > 0 else np.nan

        rows.append({
            'Retention rate': r,
            'Cost per contact': c,
            'Retained churners': retained,
            'Saved CLTV ($)': saved_cltv,
            'Saved Total Revenue ($)': saved_rev,
            'Contact cost ($)': cost,
            'Net benefit (CLTV) ($)': net_cltv,
            'Net benefit (Total Revenue) ($)': net_rev,
            'ROI (CLTV)': roi_val_cltv,
            'ROI (Total Revenue)': roi_val_rev,
        })

sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df)


,Retention rate,Cost per contact,Retained churners,Saved CLTV ($),Saved Total Revenue ($),Contact cost ($),Net benefit (CLTV) ($),Net benefit (Total Revenue) ($),ROI (CLTV),ROI (Total Revenue)
0,0.1,5,29,1.155451e+05,48668.339281,2190,1.133551e+05,46478.339281,51.760337,21.222986
1,0.1,10,29,1.155451e+05,48668.339281,4380,1.111651e+05,44288.339281,25.380168,10.111493
2,0.1,20,29,1.155451e+05,48668.339281,8760,1.067851e+05,39908.339281,12.190084,4.555746
3,0.2,5,58,2.310903e+05,97336.678562,2190,2.289003e+05,95146.678562,104.520673,43.445972
4,0.2,10,58,2.310903e+05,97336.678562,4380,2.267103e+05,92956.678562,51.760337,21.222986
5,0.2,20,58,2.310903e+05,97336.678562,8760,2.223303e+05,88576.678562,25.380168,10.111493
6,0.3,5,88,3.506197e+05,147683.236438,2190,3.484297e+05,145493.236438,159.100332,66.435268
7,0.3,10,88,3.506197e+05,147683.236438,4380,3.462397e+05,143303.236438,79.050166,32.717634
8,0.3,20,88,3.506197e+05,147683.236438,8760,3.418597e+05,138923.236438,39.025083,15.858817
9,0.4,5,117,4.661649e+05,196351.575719,2190,4.639749e+05,194161.575719,211.860668,88.658254


In [40]:
sensitivity_df_plot = sensitivity_df.copy()
sensitivity_df_plot['Cost per contact'] = sensitivity_df_plot['Cost per contact'].astype(str)

fig = px.line(
    sensitivity_df_plot.sort_values(['Cost per contact', 'Retention rate']),
    x='Retention rate',
    y='Net benefit (CLTV) ($)',
    color='Cost per contact',
    markers=True,
    title='Net Benefit (CLTV) vs Retention Rate',
    labels={
        'Retention rate': 'Retention Rate',
        'Net benefit (CLTV) ($)': 'Net Benefit (CLTV) ($)',
        'Cost per contact': 'Cost per Contact ($)',
    },
)

fig.update_layout(
    template='plotly_white',
    width=700,
    height=500,
    title_x=0.5,
    hovermode='x unified',
    legend_title_text='Cost per Contact ($)',
)

fig.update_traces(line=dict(width=3), marker=dict(size=8))

fig.update_xaxes(tickformat='.0%')
fig.update_yaxes(tickprefix='$', separatethousands=True)

fig.show()


In [41]:
fig = px.line(
    sensitivity_df_plot.sort_values(['Cost per contact', 'Retention rate']),
    x='Retention rate',
    y='Net benefit (Total Revenue) ($)',
    color='Cost per contact',
    markers=True,
    title='Net Benefit (Total Revenue) vs Retention Rate',
    labels={
        'Retention rate': 'Retention Rate',
        'Net benefit (Total Revenue) ($)': 'Net Benefit (Total Revenue) ($)',
        'Cost per contact': 'Cost per Contact ($)',
    },
)

fig.update_layout(
    template='plotly_white',
    width=700,
    height=500,
    title_x=0.5,
    hovermode='x unified',
    legend_title_text='Cost per Contact ($)',
)

fig.update_traces(line=dict(width=3), marker=dict(size=8))

fig.update_xaxes(tickformat='.0%')
fig.update_yaxes(tickprefix='$', separatethousands=True)

fig.show()

The following chart plots net benefit against retention rate, with one line per fixed contact cost. Upward-sloping lines indicate that higher retention effectiveness improves the business case. Curves that remain below zero (if any) imply that the intervention is not economically attractive under those assumptions.


In [42]:
df_sorted = df_test.sort_values('y_proba', ascending=False).reset_index(drop=True)

df_sorted['cum_customers'] = np.arange(1, len(df_sorted) + 1)
df_sorted['cum_customers_pct'] = df_sorted['cum_customers'] / len(df_sorted)

df_sorted['churned_cltv'] = np.where(df_sorted['y_true'] == 1, df_sorted['CLTV'], 0)
df_sorted['churned_revenue'] = np.where(df_sorted['y_true'] == 1, df_sorted['Total Revenue'], 0)

df_sorted['cum_churned_cltv'] = df_sorted['churned_cltv'].cumsum()
df_sorted['cum_churned_revenue'] = df_sorted['churned_revenue'].cumsum()

total_churned_cltv = df_sorted['churned_cltv'].sum()
total_churned_revenue = df_sorted['churned_revenue'].sum()

df_sorted['cum_churned_cltv_pct'] = df_sorted['cum_churned_cltv'] / total_churned_cltv
df_sorted['cum_churned_revenue_pct'] = df_sorted['cum_churned_revenue'] / total_churned_revenue


In [43]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_sorted['cum_customers_pct'],
        y=df_sorted['cum_churned_cltv_pct'],
        mode='lines',
        name='CLTV (model)',
        line=dict(width=3, color='#1f77b4'),
        hovertemplate=(
            'Targeted customers: %{x:.1%}<br>'
            'Captured churned CLTV: %{y:.1%}<extra></extra>'
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_sorted['cum_customers_pct'],
        y=df_sorted['cum_churned_revenue_pct'],
        mode='lines',
        name='Revenue (model)',
        line=dict(width=3, color='#2ca02c'),
        hovertemplate=(
            'Targeted customers: %{x:.1%}<br>'
            'Captured churned revenue: %{y:.1%}<extra></extra>'
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode='lines',
        name='Random targeting',
        line=dict(color='black', dash='dash', width=2),
        hovertemplate=(
            'Targeted customers: %{x:.1%}<br>'
            'Expected capture under random targeting: %{y:.1%}<extra></extra>'
        ),
    )
)

fig.update_layout(
    title='Cumulative Churned Revenue Capture by Model Score',
    xaxis_title='Fraction of customers targeted (highest churn probability first)',
    yaxis_title='Cumulative share of churned value captured',
    template='plotly_white',
    title_x=0.5,
    hovermode='x unified',
    legend_title_text='Curve',
    width=700,
    height=550,
)

fig.update_xaxes(tickformat='.0%')
fig.update_yaxes(tickformat='.0%')

fig.show()

**Interpretation**

This chart shows how effectively the model ranks customers by churn risk and by associated business value. Customers are sorted from highest to lowest predicted churn probability, and the curves show the cumulative share of churned CLTV and churned revenue captured as we progressively target more customers.

The dashed diagonal line represents random targeting. If the model were no better than random, targeting 20% of customers would capture about 20% of the churned CLTV or churned revenue. Curves above this line indicate that the model concentrates high-risk, high-value churners near the top of the ranking.

A steeper curve at the beginning is better because it means a smaller retention budget can capture a larger share of at-risk value. For example, if targeting the top 20% of customers captures 50% of churned CLTV, the model enables much more efficient intervention than untargeted outreach.

The difference between the CLTV and revenue curves is also informative. If the CLTV curve rises faster than the revenue curve, the model is prioritizing customers with greater long-term value even more effectively than those with high historical revenue. This is useful when retention decisions are based on future customer value rather than past revenue alone.

Overall, this plot should be interpreted as a ranking-efficiency view of the model. It does not show causal impact by itself, but it does show whether the model helps prioritize customer outreach in a way that captures more at-risk value than random targeting.

## Limitations

- This is a retrospective simulation on a static dataset, not a measured business outcome.
- CLTV and Total Revenue are observed after the fact; in a real deployment, only information available at intervention time should be used.
- The analysis assumes that retention actions can effectively reduce churn among targeted customers; actual impact depends on the effectiveness and cost of those actions.